# Qwen-Image 2.1 Uncensored · A100 40GB / 80GB

依次运行四格。**01 文字生图；02 上传图片编辑。** 默认 2K、40 步、单张，无需 LoRA。

默认 BF16 主模型 + INT8 文本编码器 + BF16 VAE。显存不足时选 1K 或 Q8_0；切换精度后重新打开工作流。


In [ ]:
# @title 1. GPU 与环境
import os, re, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("请选择带 A100 GPU 的运行时。")
gpu = torch.cuda.get_device_name(0)
memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu, f"{memory:.1f} GiB")
if "A100" not in gpu.upper() or memory < 35:
    raise RuntimeError("本笔记本面向 A100 40GB / 80GB；当前卡或显存切片不满足要求。")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("当前 CUDA / PyTorch 未启用 BF16，请切换到支持 A100 的运行时。")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
print(f"系统内存 {ram:.1f} GiB；建议至少 64GB。")

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REVISION = '7d9e5a045c13c563a5dcd18c617a985efa6332da'
GGUF_REVISION = '373048b8403a7820620065210a691263d4da0a61'
GGUF_DIR = COMFY_DIR / "custom_nodes/ComfyUI-GGUF"
QWEN_UC_INSTALLED_REVISIONS = None

def stop_previous_comfyui():
    process = globals().get("qwen_process")
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)

stop_previous_comfyui()

def install_git_revision(repository, directory, revision):
    directory = Path(directory)
    for parent in [directory, *directory.parents]:
        if parent.is_symlink():
            raise RuntimeError(f"安装路径是软链接，请使用新的 runtime：{parent}")
    fresh = not directory.exists()
    if fresh:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "--no-checkout", repository, str(directory)], check=True)
    if not (directory / ".git").is_dir():
        raise RuntimeError(f"安装目录不是 Git 仓库，请使用新的 runtime：{directory}")
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=directory, text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != repository.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"现有目录来自不同仓库，请使用新的 runtime：{directory}")
    if not fresh and subprocess.check_output(["git", "status", "--porcelain"], cwd=directory, text=True).strip():
        raise RuntimeError(f"现有仓库有未提交修改，已保留；请使用新的 runtime：{directory}")
    current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=directory, text=True).strip()
    if fresh or current != revision:
        subprocess.run(["git", "fetch", "--depth", "1", "origin", revision], cwd=directory, check=True)
        subprocess.run(["git", "checkout", "--detach", revision], cwd=directory, check=True)

install_git_revision("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_REVISION)
install_git_revision("https://github.com/leejet/ComfyUI-GGUF.git", GGUF_DIR, GGUF_REVISION)
skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for requirements_file in [COMFY_DIR / "requirements.txt", GGUF_DIR / "requirements.txt"]:
    for line in requirements_file.read_text().splitlines():
        name = re.split(r"[<>=!~\[]", line.strip(), maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(line)
requirements = Path("/content/qwen_uc_comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "huggingface_hub", "hf_xet", "ipywidgets"], cwd=COMFY_DIR, check=True)
GGUF_CHECK_SOURCE = r'''
import importlib.util
import sys
import types
from pathlib import Path
import gguf
import torch
import comfy.options

comfy.options.enable_args_parsing()
import comfy.ops

package_name = "_qwen_uc_gguf_check"
package = types.ModuleType(package_name)
package.__path__ = [str(Path("custom_nodes/ComfyUI-GGUF").resolve())]
sys.modules[package_name] = package
spec = importlib.util.spec_from_file_location(package_name + ".ops", Path(package.__path__[0]) / "ops.py")
ops = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ops
spec.loader.exec_module(ops)

weight = torch.linspace(-1, 1, 64, dtype=torch.float32).reshape(2, 32)
x = torch.linspace(-0.5, 0.5, 64, dtype=torch.float32).reshape(2, 32)
for qtype in (gguf.GGMLQuantizationType.BF16, gguf.GGMLQuantizationType.Q8_0):
    if qtype == gguf.GGMLQuantizationType.BF16:
        reference_weight = weight.to(torch.bfloat16).to(torch.float32)
        data = weight.to(torch.bfloat16).view(torch.uint8)
    else:
        encoded = gguf.quants.quantize(weight.numpy(), qtype)
        data = torch.from_numpy(encoded.copy())
        reference_weight = torch.from_numpy(gguf.quants.dequantize(encoded, qtype))
    layer = ops.GGMLOps.Linear(32, 2, bias=False)
    layer.weight = torch.nn.Parameter(ops.GGMLTensor(data, tensor_type=qtype, tensor_shape=weight.shape), requires_grad=False)
    with torch.inference_mode():
        actual = layer(x)
        expected = torch.nn.functional.linear(x, reference_weight)
    torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-5)
print("GGUF BF16 / Q8_0 检查通过")
'''
subprocess.run([sys.executable, "-c", GGUF_CHECK_SOURCE, "--disable-dynamic-vram", "--use-pytorch-cross-attention"], cwd=COMFY_DIR, check=True)
QWEN_UC_INSTALLED_REVISIONS = (COMFY_REVISION, GGUF_REVISION)
print("ComfyUI:", COMFY_REVISION[:12], "| ComfyUI-GGUF:", GGUF_REVISION[:12])


In [ ]:
# @title 2. 模型与 GitHub 工作流
主模型精度 = "BF16" # @param ["BF16", "Q8_0"]
文本编码器精度 = "INT8" # @param ["INT8", "BF16"]
WORKFLOW_REF = "main" # @param {type:"string"}

import copy, hashlib, json, os, shutil, tempfile, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import quote
from huggingface_hub import hf_hub_download

MODEL_REPO = 'abenzerps/Qwen-Image-2.1-Uncensored-GGUF'
MODEL_REVISION = '6b34e59458d3eb7ba6a6f86a116aed5253dc02c3'
MAIN_MODELS = {'BF16': {'remote': 'qwen-image-2.1-UC-BF16.gguf', 'file': 'diffusion_models/qwen-image-2.1-UC-BF16.gguf', 'size': 14230272800, 'sha256': 'f151c683a8aed4b310777017ebbbe3f2180f1180f7867115171adb7d50b0762a'}, 'Q8_0': {'remote': 'qwen-image-2.1-UC-Q8_0.gguf', 'file': 'diffusion_models/qwen-image-2.1-UC-Q8_0.gguf', 'size': 7591557920, 'sha256': 'cde456c72ea3ecebfc1be783300e972711d875e0c5f1bed33d42b66b156affa8'}}
TEXT_ENCODERS = {'INT8': {'remote': 'text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'file': 'text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'size': 9350798360, 'sha256': '8bfd0f6e12abf2d2d697ecc888e5e90b0d6741d6708f05799f53afa560452e8f'}, 'BF16': {'remote': 'text_encoders/qwen3vl_8b_bf16.safetensors', 'file': 'text_encoders/qwen3vl_8b_bf16.safetensors', 'size': 17534334616, 'sha256': '68bdc82bc1b66851162ae656225e7e2068166b603db19bd5d5a3b90eb12669a9'}}
VAE_MODEL = {'remote': 'vae/qwen_image_2.1_vae_bf16.safetensors', 'file': 'vae/qwen_image_2.1_vae_bf16.safetensors', 'size': 675509688, 'sha256': 'bb21f7473051e1ac368515dd3f2e15cd44d7a11748ee8823e1ddca3e4876b7c9'}
MODEL_FILES = [MAIN_MODELS[主模型精度], TEXT_ENCODERS[文本编码器精度], VAE_MODEL]
MODEL_DIR = COMFY_DIR / "models"
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows"
WORKFLOW_REPO = "boltguo/colab"
WORKFLOW_FOLDER = "workflows/qwen-image-2.1-uncensored"
WORKFLOW_FILES = (
    "01_Qwen_Image_2_1_Uncensored_T2I_A100.json",
    "02_Qwen_Image_2_1_Uncensored_Reference_A100.json",
    "03_Qwen_Image_2_1_Uncensored_Two_Images_A100.json",
)
SELECTED_MODELS = {
    "UnetLoaderGGUF": MAIN_MODELS[主模型精度],
    "CLIPLoader": TEXT_ENCODERS[文本编码器精度],
    "VAELoader": VAE_MODEL,
}

def local_directory(path):
    path = Path(path)
    if not path.is_relative_to(COMFY_DIR):
        raise ValueError("目录必须位于 ComfyUI 中。")
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def valid_model(path, item):
    return (path.is_file() and not path.is_symlink() and path.stat().st_size == item["size"]
            and sha256_file(path) == item["sha256"])

def download_models(manifest, directory):
    local_directory(directory)
    missing = 0
    for item in manifest:
        target = directory / item["file"]
        local_directory(target.parent)
        if target.is_symlink() or (target.exists() and not target.is_file()):
            raise RuntimeError(f"模型目标路径无效：{target}")
        if target.exists():
            print("校验已有模型：", target.name)
            if not valid_model(target, item):
                raise RuntimeError(f"已有模型大小或 SHA256 不匹配，已保留；检查或移走后重跑：{target}")
        else:
            missing += item["size"]
    if shutil.disk_usage(directory).free < missing + 2 * 1024**3:
        raise RuntimeError(f"磁盘空间不足，本次需下载约 {missing / 10**9:.2f} GB，另需 2GiB 余量。")
    for item in manifest:
        target = directory / item["file"]
        if target.exists():
            print("已存在：", target.name)
            continue
        # GGUF is stored at the HF repository root, but ComfyUI searches diffusion_models.
        download_root = directory / "diffusion_models" if "/" not in item["remote"] else directory
        result = Path(hf_hub_download(repo_id=MODEL_REPO, filename=item["remote"], revision=MODEL_REVISION,
                                     local_dir=str(download_root)))
        if result.resolve() != target.resolve() or not valid_model(target, item):
            raise RuntimeError(f"下载路径、大小或 SHA256 校验失败：{target}")
        print("下载并校验完成：", target.name)

def all_workflow_nodes(data):
    yield from data["nodes"]
    for graph in data.get("definitions", {}).get("subgraphs", []):
        yield from graph["nodes"]

def configure_workflow(data):
    data = copy.deepcopy(data)
    if not isinstance(data, dict) or not isinstance(data.get("nodes"), list) or not data["nodes"] or not isinstance(data.get("links"), list):
        raise ValueError("工作流 JSON 缺少 nodes 或 links。")
    counts = dict.fromkeys(SELECTED_MODELS, 0)
    for node in all_workflow_nodes(data):
        kind = node.get("type")
        if not isinstance(kind, str):
            raise ValueError("工作流节点结构无效。")
        if "lora" in kind.lower():
            raise ValueError("这些基础工作流不应包含 LoRA 节点。")
        if kind == "UNETLoader":
            raise ValueError("工作流必须使用 UnetLoaderGGUF。")
        if kind not in SELECTED_MODELS:
            continue
        counts[kind] += 1
        item = SELECTED_MODELS[kind]
        filename = Path(item["file"]).name
        values = node.get("widgets_values")
        if not isinstance(values, list) or not values:
            raise ValueError(f"{kind} 缺少模型选择。")
        values[0] = filename
        if kind == "CLIPLoader":
            node["widgets_values"] = [filename, "qwen_image", "default"]
            node["title"] = "文本编码器 · " + ("INT8 ConvRot" if 文本编码器精度 == "INT8" else "BF16")
        elif kind == "UnetLoaderGGUF":
            node["widgets_values"] = [filename]
            node["title"] = "主模型 · Uncensored " + 主模型精度 + " GGUF"
        node.setdefault("properties", {})["models"] = [{
            "name": filename, "directory": str(Path(item["file"]).parent),
            "url": f"https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{item['remote']}",
        }]
    if any(count != 1 for count in counts.values()):
        raise ValueError("工作流应各包含一个 GGUF 主模型、文本编码器和 VAE 加载节点。")
    metadata = data.setdefault("extra", {}).setdefault("qwen_workflow", {})
    metadata.update(model_repo=MODEL_REPO, model_revision=MODEL_REVISION,
                    precision={"diffusion": 主模型精度, "text_encoder": 文本编码器精度, "vae": "BF16"})
    return data

def read_github_workflow(name):
    ref = WORKFLOW_REF.strip()
    if not ref:
        raise ValueError("WORKFLOW_REF 不能为空。")
    url = f"https://raw.githubusercontent.com/{WORKFLOW_REPO}/{quote(ref, safe='')}/{WORKFLOW_FOLDER}/{quote(name, safe='')}"
    request = urllib.request.Request(url, headers={"User-Agent": "Qwen-UC-Colab-Workflow-Downloader"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read(2_000_001)
        if len(body) > 2_000_000 or "text/html" in response.headers.get("Content-Type", "").lower():
            raise ValueError("GitHub 返回网页或文件超过 2MB。")
    return configure_workflow(json.loads(body))

def install_workflow(name, data):
    if name not in WORKFLOW_FILES or Path(name).name != name:
        raise ValueError("工作流文件名无效。")
    data = configure_workflow(data)
    local_directory(WORKFLOW_DIR)
    target = WORKFLOW_DIR / name
    if target.is_symlink() or (target.exists() and not target.is_file()):
        raise ValueError("目标工作流是软链接或目录，未写入。")
    fd, temporary = tempfile.mkstemp(prefix=".qwen-uc-workflow-", dir=WORKFLOW_DIR)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
            file.write("\n")
        os.replace(temporary, target)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)
    print("工作流已安装：", name)

def download_github_workflows():
    for name in WORKFLOW_FILES:
        try:
            install_workflow(name, read_github_workflow(name))
        except (urllib.error.URLError, OSError, ValueError, TypeError, KeyError) as error:
            print(f"工作流 {name} 下载失败：{error}；保留已有文件。")

for path in [MODEL_DIR, COMFY_DIR / "input", COMFY_DIR / "output", WORKFLOW_DIR]:
    local_directory(path)
total_size = sum(item["size"] for item in MODEL_FILES)
print(f"选定模型：{主模型精度} 主模型 / {文本编码器精度} 文本编码器 / BF16 VAE；文件合计 {total_size / 10**9:.2f} GB。")
download_models(MODEL_FILES, MODEL_DIR)
download_github_workflows()


In [ ]:
# @title 3. 启动并打开 ComfyUI
import json, subprocess, sys, time, urllib.request
from pathlib import Path
from google.colab import output
from html import escape
from urllib.parse import urlsplit
from IPython.display import HTML, display

COMFY_DIR = Path("/content/ComfyUI")
BASE = "http://127.0.0.1:8188"
LOG = Path("/content/qwen_uc_comfyui.log")
def get_json(route):
    with urllib.request.urlopen(BASE + route, timeout=3) as response:
        return json.load(response)
def ready():
    try:
        get_json("/system_stats")
        return True
    except Exception:
        return False

LAUNCHER = Path("/content/qwen_uc_comfy_userdata_compat.py")
LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases for decoded proxy paths."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; compatibility aliases were not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            # Preserve the original user checks, safe response headers,
            # atomic writes, overwrite flags, and deletion semantics.
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI source directory is missing or invalid.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''
LAUNCHER.write_text(LAUNCHER_SOURCE)
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8188)")
if not isinstance(proxy_url, str) or not proxy_url.startswith("https://"):
    raise ValueError("Colab 未返回 HTTPS 代理链接。")
proxy_parts = urlsplit(proxy_url)
proxy_origin = f"{proxy_parts.scheme}://{proxy_parts.netloc}"

if globals().get("QWEN_UC_INSTALLED_REVISIONS") != (globals().get("COMFY_REVISION"), globals().get("GGUF_REVISION")):
    raise RuntimeError("请先运行第 1 格安装并检查环境。")
if ready() and (globals().get("qwen_process") is None or qwen_process.poll() is not None
                or globals().get("QWEN_UC_SERVER_REVISIONS") != QWEN_UC_INSTALLED_REVISIONS):
    raise RuntimeError("8188 端口已有旧服务，请停止该服务后重新运行第 3 格。")

if not ready():
    with LOG.open("w") as log:
        qwen_process = subprocess.Popen([sys.executable, str(LAUNCHER), "--listen", "127.0.0.1", "--port", "8188", "--output-directory", str(COMFY_DIR / "output"), "--enable-cors-header", proxy_origin, "--disable-all-custom-nodes", "--whitelist-custom-nodes", "ComfyUI-GGUF", "--disable-partner-nodes", "--bf16-unet", "--bf16-vae", "--disable-dynamic-vram", "--use-pytorch-cross-attention"], cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.time() + 600
    while time.time() < deadline:
        if qwen_process.poll() is not None:
            print(LOG.read_text(errors="replace")[-8000:])
            raise RuntimeError("ComfyUI 启动失败，请查看上方日志。")
        if ready():
            break
        time.sleep(2)
    else:
        raise TimeoutError("ComfyUI 10 分钟内未就绪。")
QWEN_UC_SERVER_REVISIONS = QWEN_UC_INSTALLED_REVISIONS
registered = get_json("/object_info")
if not {"UnetLoaderGGUF", "TextEncodeQwenImage21", "QwenImage21Cache", "CustomCombo", "ComfyMathExpression", "ResolutionSelector"}.issubset(registered):
    raise RuntimeError("缺少 GGUF / Qwen / 尺寸节点。若复用了之前启动的 ComfyUI，请断开并删除运行时后从第 1 格重跑；否则检查第 1 格安装与下方日志。日志：/content/qwen_uc_comfyui.log。")
gguf_names = registered["UnetLoaderGGUF"]["input"]["required"]["unet_name"][0]
clip_names = registered["CLIPLoader"]["input"]["required"]["clip_name"][0]
vae_names = registered["VAELoader"]["input"]["required"]["vae_name"][0]
for kind, names in [("UnetLoaderGGUF", gguf_names), ("CLIPLoader", clip_names), ("VAELoader", vae_names)]:
    expected = Path(SELECTED_MODELS[kind]["file"]).name
    if expected not in names:
        raise RuntimeError(f"ComfyUI 未发现模型 {expected}，请检查第 2 格下载。")
display(HTML(f'<a href="{escape(proxy_url, quote=True)}" target="_blank" rel="noopener">新窗口打开 ComfyUI</a>'))
print("打开 01_Qwen_Image_2_1_Uncensored_T2I_A100：输入文字生成图片。")
print("02 为单图编辑，03 为双图编辑；先上传图片，再填写编辑指令。")
print("2K / 40 步 / 单张；无需 LoRA。")
print("生成后运行第 4 格下载；重跑该格刷新列表。")


In [ ]:
# @title 4. 下载图片与工作流
import json, os, tempfile, zipfile
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from google.colab import files

OUTPUT_DIR = Path("/content/ComfyUI/output/qwen-image-2.1-uncensored")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
def result_files(directory):
    directory = Path(directory)
    if directory.is_symlink():
        raise ValueError("输出目录是软链接，请核对保存目录。")
    root = directory.resolve()
    results = []
    for base, folders, names in os.walk(directory, followlinks=False):
        folders[:] = [name for name in folders if not (Path(base) / name).is_symlink()]
        for name in names:
            path = Path(base) / name
            if path.suffix.lower() in IMAGE_EXTENSIONS and not path.is_symlink() and path.is_file() and path.resolve().is_relative_to(root):
                results.append(path)
    return sorted(results, key=lambda path: path.stat().st_mtime, reverse=True)

def make_result_zip(selected, directory, archive):
    selected, directory, archive = Path(selected), Path(directory), Path(archive)
    if selected not in result_files(directory) or not selected.resolve().is_relative_to(directory.resolve()):
        raise ValueError("请选择列表中的图片。")
    if archive.is_symlink() or archive.resolve().is_relative_to(directory.resolve()):
        raise ValueError("ZIP 须保存在 output 外，且不能是软链接。")
    workflow = None
    try:
        with Image.open(selected) as image:
            embedded = image.info.get("workflow")
        if embedded:
            data = json.loads(embedded) if isinstance(embedded, str) else embedded
            if isinstance(data, dict) and isinstance(data.get("nodes"), list) and isinstance(data.get("links"), list):
                workflow = data
    except Exception as error:
        print("无法读取内嵌工作流，仍下载图片：", error)
    fd, temporary = tempfile.mkstemp(prefix=".qwen-result-", suffix=".zip", dir=archive.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as bundle:
            if selected.is_symlink() or not selected.is_file() or not selected.resolve().is_relative_to(directory.resolve()):
                raise ValueError("图片路径已变化，未完成打包。")
            bundle.write(selected, arcname=selected.name)
            if workflow is not None:
                bundle.writestr("workflow.json", json.dumps(workflow, ensure_ascii=False, indent=2))
        os.replace(temporary, archive)
        return archive
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def make_all_images_zip(directory, archive):
    directory, archive = Path(directory), Path(archive)
    results = result_files(directory)
    if not results:
        raise ValueError("输出目录及子目录内暂无 PNG、JPG、JPEG 或 WEBP 图片。")
    if archive.is_symlink() or archive.resolve().is_relative_to(directory.resolve()):
        raise ValueError("ZIP 须保存在 output 外，且不能是软链接。")
    fd, temporary = tempfile.mkstemp(prefix=".qwen-output-", suffix=".zip", dir=archive.parent)
    os.close(fd)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as bundle:
            for path in results:
                if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(directory.resolve()):
                    raise ValueError("图片路径已变化，未完成打包。")
                bundle.write(path, arcname=path.relative_to(directory).as_posix())
        os.replace(temporary, archive)
        return archive
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

results = result_files(OUTPUT_DIR)
print("输出目录：", OUTPUT_DIR, "；找到图片：", len(results))
if not results:
    print("暂无图片。生成后重跑本格；已有结果请核对输出目录。")
else:
    selector = widgets.Dropdown(options=[(str(path.relative_to(OUTPUT_DIR)), str(path)) for path in results], description="图片", layout=widgets.Layout(width="95%"))
    button = widgets.Button(description="下载所选图片＋工作流")
    all_button = widgets.Button(description="下载全部图片")
    status = widgets.Output()
    def download(_):
        with status:
            status.clear_output()
            try:
                archive = make_result_zip(selector.value, OUTPUT_DIR, Path("/content/qwen_uc_result.zip"))
                files.download(str(archive))
            except Exception as error:
                print("下载失败：", error)
    def download_all(_):
        with status:
            status.clear_output()
            try:
                archive = make_all_images_zip(OUTPUT_DIR, Path("/content/qwen_uc_all_output_images.zip"))
                print("全部图片已打包。")
                files.download(str(archive))
            except Exception as error:
                print("下载全部图片失败：", error)
    button.on_click(download)
    all_button.on_click(download_all)
    display(selector, widgets.HBox([button, all_button]), status)


[模型](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF) · [GGUF 节点](https://github.com/leejet/ComfyUI-GGUF) · [模型许可](https://huggingface.co/Qwen/Qwen-Image-2.1/blob/main/LICENSE)。Built with Qwen.
